# 01 - Data exploration

The case-study record, its quality, its daylight structure and the derived
features. Run after `python scripts/preprocess.py`; every plot and number
comes from the processed artefacts in `data/processed/`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)


In [ ]:
featured = pd.read_parquet(ROOT / "data" / "processed" /
                          "featured_primary_station.parquet")
print(featured.shape)
display(featured.head())


In [ ]:
# Half the steps are exactly zero: night-time. This is why MAPE is not
# computed and why daylight and all-step metrics are reported separately.
display(featured["pv_power_w"].describe())
print("zero fraction:", float((featured["pv_power_w"] <= 0).mean()))


In [ ]:
rated = featured["pv_power_w"].max()
times = pd.to_datetime(featured["Time"])
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
axes[0].plot(times, featured["pv_power_w"] / 1000, lw=0.4,
            color="#1f77b4")
axes[0].set_ylabel("PV power (kW)")
axes[0].set_title(f"Case-study station, full record; peak "
                    f"{rated / 1000:.1f} kW")
axes[1].hist(featured.loc[featured["daylight"], "pv_power_w"] / 1000,
            bins=80, color="#ff7f0e")
axes[1].set_xlabel("Daylight power (kW)")
axes[1].set_ylabel("Count")
plt.tight_layout()


In [ ]:
# Class balance of the weather-regime taxonomy, with the persistence
# reference rescored inside each stratum in the analysis stage.
display(featured["regime"].value_counts())


In [ ]:
from solar_forecasting.features.builder import (feature_columns,
                                            describe_feature_groups)
columns = feature_columns(featured)
print(len(columns), "model inputs")
for name, group in describe_feature_groups(columns).items():
    print(f"{name:>18}: {len(group):>3}  {group[:6]}")


## Reading the exploration

- The zero fraction is the single most important number here: it governs
  the metric choice and the evaluation scope.
- The regime counts are the class-balance check for the stratified
  analysis. A stratum with fewer than 200 daylight steps is reported as
  under-sampled rather than summarised.
